In [31]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
from torch import nn

In [5]:
SMALL_CAP = [
    "600169.SH", "000049.SZ", "600428.SH", "002064.SZ", "000501.SZ", "603000.SH", "000712.SZ", "600580.SH", "600017.SH", "603355.SH"
]

MID_CAP = [
    "600138.SH", "000718.SZ", "002444.SZ", "600597.SH", "002285.SZ", "600729.SH", "000062.SZ", "002463.SZ", "002603.SZ", "603883.SH"
]

LARGE_CAP = [
    "600718.SH", "600507.SH", "002583.SZ", "601718.SH", "600835.SH", "002268.SZ", "002092.SZ", "600098.SH", "002340.SZ", "300001.SZ"
]

TICKERS = SMALL_CAP + MID_CAP + LARGE_CAP

In [27]:
DATA = Path("./data")

# Prices and returns
daily_prices = pd.read_pickle(DATA / "daily_adj_close_post_20260630.pkl")
monthly_returns = pd.read_pickle(DATA / "monthly_returns.pkl")
prices = pd.read_hdf(DATA / "price.h5", key="price")

# Trading characteristics
daily_amount = pd.read_pickle(DATA / "daily_amount.pkl")
daily_turnover = pd.read_pickle(DATA / "daily_turnover.pkl")
daily_volume = pd.read_pickle(DATA / "daily_volume.pkl")

# Company characteristics
mcap = pd.read_pickle(DATA / "mcap.pkl")
industry = pd.read_pickle(DATA / "ind_code.pkl")

# Factor returns
ff5 = pd.read_csv(DATA / "FF5.csv", index_col=0, parse_dates=True)
hxz = pd.read_csv(DATA / "HXZ.csv", index_col=0, parse_dates=True)

# Risk-free rates
risk_free = pd.read_csv(DATA / "risk_free.csv")
shibor = pd.read_pickle(DATA / "shibor_rates.pkl")

# Stock selection and trading constraints
blacklist = pd.read_pickle(DATA / "BLACKLIST.pkl")
untradable = pd.read_pickle(DATA / "UNTRADABLE.pkl")
csi500_mask = pd.read_pickle(DATA / "csi500_mask_monthly.pkl")

In [18]:
prices

asset,000001.SZ,000002.SZ,000003.SZ,000004.SZ,000005.SZ,000006.SZ,000007.SZ,000008.SZ,000009.SZ,000010.SZ,...,873693.BJ,873703.BJ,873706.BJ,873726.BJ,873806.BJ,873833.BJ,920002.BJ,920008.BJ,920118.BJ,T00018.SH
date,,,,,,,,,,,,,,,,,,,,,
1995-01-03,60.52512,23.82854,19.12960,17.13948,10.96780,13.60100,9.92006,8.85000,9.98455,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-04,63.00936,24.17612,19.41184,17.69868,11.45326,13.86364,10.27246,9.04175,10.08380,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-05,62.78352,23.75130,19.12960,17.25132,11.27346,13.73232,10.27246,8.95325,9.96470,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-06,63.00936,23.86716,19.22368,17.47500,11.29144,13.78860,10.30770,8.96800,9.96470,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-09,60.92034,23.24924,18.78464,16.80396,10.93184,13.31960,9.83196,8.61400,9.68680,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-26,1311.76401,1186.52712,NaN,47.46752,NaN,154.98600,35.78688,44.59192,68.49451,18.53300,...,21.131650,22.73188,9.462684,15.704568,7.775232,8.372148,44.97,NaN,19.68,NaN
2024-08-27,1304.26107,1146.55224,NaN,47.95520,NaN,149.81980,37.02948,43.91968,67.14796,18.20975,...,20.220285,23.26320,9.349494,15.704568,7.441140,8.216076,43.31,NaN,18.25,NaN
2024-08-28,1291.75617,1130.19888,NaN,47.22368,NaN,149.81980,39.10048,44.14376,67.50704,18.20975,...,20.721035,23.40680,9.372132,15.829704,7.441140,8.249520,44.03,NaN,19.12,NaN


In [22]:
WINDOW = 504
OOS_START = "2019-01-02"
OOS_END = "2024-08-30"

prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()

selected_prices = prices[TICKERS].copy()

selected_prices

asset,600169.SH,000049.SZ,600428.SH,002064.SZ,000501.SZ,603000.SH,000712.SZ,600580.SH,600017.SH,603355.SH,...,600718.SH,600507.SH,002583.SZ,601718.SH,600835.SH,002268.SZ,002092.SZ,600098.SH,002340.SZ,300001.SZ
date,,,,,,,,,,,,,,,,,,,,,
1995-01-03,NaN,NaN,NaN,NaN,5.767680,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,12.85200,NaN,NaN,NaN,NaN,NaN
1995-01-04,NaN,NaN,NaN,NaN,5.902860,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,13.05360,NaN,NaN,NaN,NaN,NaN
1995-01-05,NaN,NaN,NaN,NaN,6.158200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,12.81168,NaN,NaN,NaN,NaN,NaN
1995-01-06,NaN,NaN,NaN,NaN,6.188240,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,12.73104,NaN,NaN,NaN,NaN,NaN
1995-01-09,NaN,NaN,NaN,NaN,5.947920,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,12.49920,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-26,18.66496,215.08657,37.208532,87.613745,46.688950,74.898551,132.3896,79.804725,9.67561,38.04840,...,132.592572,95.75292,21.74338,2.741375,63.26510,101.507315,12.30592,36.263620,141.653508,126.20560
2024-08-27,18.46208,201.93879,37.345580,88.223020,46.898005,73.252429,134.4952,76.801725,9.67561,37.36395,...,130.678506,95.50160,21.12375,2.730000,62.58422,99.877020,12.13112,36.263620,139.206988,119.70304
2024-08-28,18.56352,203.32764,37.277056,87.857455,48.570445,74.811913,132.7844,76.876800,9.71054,37.38150,...,131.548536,95.75292,21.23641,2.752750,62.01682,100.048630,12.20104,36.263620,139.696292,120.05264
